# PALS: Probing Alignment in Language-model Sycophancy — Full 16-Hypothesis Suite on OLMo-3 7B

This notebook runs the complete PALS experiment suite on **OLMo-3 7B** using Google Colab with a T4 GPU.

**Three research themes:**

| Theme | Hypotheses | Question |
|-------|-----------|----------|
| A. Representation geometry | H1, H2, H6, H7, H9, H10, H11, H13, H15 | How does the model *represent* clinical vs. factual sycophancy? |
| B. Training dynamics | H3, H5, H8 | When and how does alignment training reshape sycophancy circuits? |
| C. Intervention & validation | H4, H12, H14, H16 | Can we *steer* or *attribute* sycophancy at generation time? |

**Execution order:**
- **Phase 1** (single DPO model): H1, H2, H4, H6, H7, H9, H10, H11, H12, H13, H14, H15, H16
- Free DPO model memory
- **Phase 2** (multi-checkpoint): H3, H5, H8 (each loads its own models)

Designed for **Run All** — no manual intervention needed.

In [ ]:
# === Setup: Clone repo, install dependencies, check GPU ===
import subprocess, sys, os

if not os.path.exists("/content/pals"):
    subprocess.run(["git", "clone", "https://github.com/elliottleow/pals.git", "/content/pals"], check=True)

os.chdir("/content/pals")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Check GPU
import torch
print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# === Verify stimuli exist ===
import json, os

STIMULI_DIR = "/content/pals/stimuli"

required_stimuli = [
    "cognitive_distortions.json",
    "clinical_bridge.json",
    "factual_control.json",
    "high_emotion_general.json",
    "clinical_correct_answer.json",
    "emotional_intensity_gradient.json",
]

missing = [f for f in required_stimuli if not os.path.exists(os.path.join(STIMULI_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing stimulus files: {missing}")

for f in required_stimuli:
    with open(os.path.join(STIMULI_DIR, f)) as fh:
        data = json.load(fh)
    print(f"  {f}: {len(data)} items")

print("\nAll stimuli verified.")

In [ ]:
# === Configuration: imports, constants, helpers ===
import gc
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from glob import glob

import torch
from IPython.display import display, Image

sys.path.insert(0, "/content/pals/src")
sys.path.insert(0, "/content/pals")

from pals.models import load_model, load_tokenizer, get_device

# =====================================================================
# >>> CHANGE THIS TO SWITCH BETWEEN 7B AND 32B <<<
# =====================================================================
USE_32B = False  # Set True for OLMo-3 32B (requires A100 40GB+ with 4-bit)
# =====================================================================

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if USE_32B:
    QUANTIZE = True  # 32B requires 4-bit even on A100 40GB — results are a scaling check, not primary
    CHECKPOINTS = {
        "base": "allenai/Olmo-3-1125-32B",
        "sft": "allenai/Olmo-3.1-32B-Instruct-SFT",
        "dpo": "allenai/Olmo-3.1-32B-Instruct-DPO",
    }
    DPO_MODEL_ID = CHECKPOINTS["dpo"]
    OUTPUT_DIR = "/content/pals/results/olmo3_32b"
    # Reduce stimuli counts — 32B is ~4x slower per forward pass
    N_STIMULI_H4 = 5
    N_STIMULI_H12 = 5
    N_STIMULI_H14 = 5
    N_GENERATE_H16 = 3
else:
    QUANTIZE = False  # 7B fits in fp16 on T4 (~14GB) — no quantization needed
    CHECKPOINTS = {
        "base": "allenai/Olmo-3-1025-7B",
        "sft": "allenai/Olmo-3-7B-Instruct-SFT",
        "dpo": "allenai/Olmo-3-7B-Instruct-DPO",
    }
    DPO_MODEL_ID = CHECKPOINTS["dpo"]
    OUTPUT_DIR = "/content/pals/results/olmo3_7b"
    N_STIMULI_H4 = 10
    N_STIMULI_H12 = 10
    N_STIMULI_H14 = 10
    N_GENERATE_H16 = 5

# --- Paths ---
STIMULI_DIR = "/content/pals/stimuli"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- Helper functions ---
def free_model(*models):
    """Delete model(s) and free GPU memory."""
    for m in models:
        if m is not None:
            del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def show_plots(output_dir):
    """Display all PNG plots from an experiment output directory."""
    pngs = sorted(glob(os.path.join(output_dir, "*.png")))
    if not pngs:
        print("(no plots generated)")
        return
    for p in pngs:
        print(os.path.basename(p))
        display(Image(filename=p))


def build_metadata(model_id, device, model=None):
    """Build a metadata dict for result injection."""
    meta = {
        "model": model_id,
        "device": device,
        "platform": platform.platform(),
        "torch_version": torch.__version__,
    }
    if model is not None:
        meta["num_layers"] = model.config.num_hidden_layers
        meta["hidden_size"] = model.config.hidden_size
        meta["num_params_M"] = round(sum(p.numel() for p in model.parameters()) / 1e6)
    return meta


def inject_metadata(result_json_path, metadata):
    """Read a result JSON, add metadata, and re-write it."""
    with open(result_json_path) as f:
        data = json.load(f)
    data["metadata"] = metadata
    with open(result_json_path, "w") as f:
        json.dump(data, f, indent=2)


model_label = "OLMo-3 32B (4-bit)" if USE_32B else "OLMo-3 7B"
print(f"Model: {model_label}")
print(f"Device: {DEVICE}")
print(f"Quantize: {QUANTIZE}")
print(f"Output: {OUTPUT_DIR}")
print(f"DPO model: {DPO_MODEL_ID}")
print(f"Checkpoints: {list(CHECKPOINTS.keys())}")
if USE_32B:
    print(f"\n*** 32B MODE: reduced stimuli (H4={N_STIMULI_H4}, H12={N_STIMULI_H12}, H14={N_STIMULI_H14}, H16 gen={N_GENERATE_H16})")


In [ ]:
# === Load DPO model for Phase 1 experiments ===
print(f"Loading {DPO_MODEL_ID} on {DEVICE} (4-bit={QUANTIZE})...")
t0 = time.time()
model = load_model(DPO_MODEL_ID, device=DEVICE, quantize_4bit=QUANTIZE)
tokenizer = load_tokenizer(DPO_MODEL_ID)
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"  Architecture: {model.config.model_type}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Hidden dim: {model.config.hidden_size}")
print(f"  Attention heads: {model.config.num_attention_heads}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"  Device: {get_device(model)}")

---
## Phase 1: Single-Model Experiments (DPO checkpoint)

---

### H1: Is clinical sycophancy a different direction from factual sycophancy?

Extract contrastive directions for clinical, bridge, and factual sycophancy. Measure cosine similarity and cross-domain probe transfer.

**OLMo-2 1B pilot:** Clinical-factual cosine similarity ~0.35 at peak, significantly below 1.0. Cross-domain probe transfer was asymmetric (factual probe generalizes to clinical better than reverse), suggesting clinical sycophancy *contains* factual sycophancy plus additional dimensions.

In [ ]:
# === H1: Direction Comparison ===
from experiments.h1_direction_comparison import run as run_h1

h1_output = os.path.join(OUTPUT_DIR, "h1")
t0 = time.time()
run_h1(model, tokenizer, STIMULI_DIR, h1_output)
elapsed = time.time() - t0
print(f"\nH1 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h1_output, "h1_results.json"), meta)
show_plots(h1_output)

### H2: Is clinical sycophancy driven by uncertainty or social deference?

Logit lens at each layer to check whether the model internally predicts the correct answer before overriding with sycophancy.

**OLMo-2 1B pilot:** Correct-answer signal peaked at +5.3 in layer 11 for factual errors and +3.1 for clinical items, suggesting the model *knows* the right answer but overrides it (deference, not uncertainty).

In [ ]:
# === H2: Uncertainty vs Deference ===
from experiments.h2_uncertainty_deference import run as run_h2

h2_output = os.path.join(OUTPUT_DIR, "h2")
t0 = time.time()
run_h2(model, tokenizer, STIMULI_DIR, h2_output)
elapsed = time.time() - t0
print(f"\nH2 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h2_output, "h2_results.json"), meta)
show_plots(h2_output)

### H4: The Representation-Generation Dissociation

If the model knows the correct answer but still generates sycophancy, specific attention heads in late layers may route generation away from the correct representation.

**OLMo-2 1B pilot:** Found 3-5 "sycophancy routing heads" in layers 12-15 whose ablation shifted output toward therapeutic. Partial overlap with factual sycophancy routing heads.

In [ ]:
# === H4: Attention Head Routing ===
from experiments.h4_attention_routing import run as run_h4

h4_output = os.path.join(OUTPUT_DIR, "h4")
t0 = time.time()
run_h4(model, tokenizer, STIMULI_DIR, h4_output, n_stimuli=N_STIMULI_H4)
elapsed = time.time() - t0
print(f"\nH4 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h4_output, "h4_results.json"), meta)
show_plots(h4_output)

### H6: Emotional Intensity Monotonically Modulates Sycophancy Mechanism

The same false belief at varying emotional intensity should show a gradient of cosine similarity with the factual sycophancy direction.

**OLMo-2 1B pilot:** Monotonically decreasing cosine similarity (factual alignment) as emotion increases: low=0.42, medium=0.31, high=0.18. Emotion pushes the mechanism away from factual sycophancy.

In [ ]:
# === H6: Emotional Intensity Gradient ===
from experiments.h6_emotional_gradient import run as run_h6

h6_output = os.path.join(OUTPUT_DIR, "h6")
t0 = time.time()
run_h6(model, tokenizer, STIMULI_DIR, h6_output)
elapsed = time.time() - t0
print(f"\nH6 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h6_output, "h6_results.json"), meta)
show_plots(h6_output)

### H7: Distortion-Specific Subspaces Within Clinical Sycophancy

Different cognitive distortion types may have different representational signatures. Compute pairwise cosine similarity and hierarchical clustering.

**OLMo-2 1B pilot:** Distortion types clustered into 2-3 families. Catastrophizing was closest to factual sycophancy (cos ~0.5), while personalization was most distant (cos ~0.15).

In [ ]:
# === H7: Distortion Subspaces ===
from experiments.h7_distortion_subspaces import run as run_h7

h7_output = os.path.join(OUTPUT_DIR, "h7")
t0 = time.time()
run_h7(model, tokenizer, STIMULI_DIR, h7_output)
elapsed = time.time() - t0
print(f"\nH7 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h7_output, "h7_results.json"), meta)
show_plots(h7_output)

### H9: The Warmth Tax — Does Clinical Sycophancy Reduce to Empathy + Agreement?

Decompose the clinical sycophancy direction into empathy + factual sycophancy + unique residual.

**OLMo-2 1B pilot:** Empathy explained ~10%, factual sycophancy ~7%, residual ~83%. Clinical sycophancy is NOT simply empathy + agreement — the unique component dominates.

In [ ]:
# === H9: Warmth Tax ===
from experiments.h9_warmth_tax import run as run_h9

h9_output = os.path.join(OUTPUT_DIR, "h9")
t0 = time.time()
run_h9(model, tokenizer, STIMULI_DIR, h9_output)
elapsed = time.time() - t0
print(f"\nH9 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h9_output, "h9_results.json"), meta)
show_plots(h9_output)

### H10: Extended Decomposition — What IS the 83% Residual?

Decompose clinical sycophancy against 5 reference directions: empathy, factual sycophancy, conflict-avoidance, clinical warmth, framing acceptance.

**OLMo-2 1B pilot:** Even with 5 reference directions, ~63% residual remained. Clinical warmth and conflict-avoidance each contributed ~5-8%. The residual is genuinely novel.

In [ ]:
# === H10: Extended Decomposition ===
from experiments.h10_extended_decomposition import run as run_h10

h10_output = os.path.join(OUTPUT_DIR, "h10")
t0 = time.time()
run_h10(model, tokenizer, STIMULI_DIR, h10_output)
elapsed = time.time() - t0
print(f"\nH10 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h10_output, "h10_results.json"), meta)
show_plots(h10_output)

### H11: Orthogonal Complement — What Do Clinical's Extra Dimensions Encode?

Project out the factual direction and analyze the residual. Does it predict emotional intensity? Distortion type? How does it relate to empathy?

**OLMo-2 1B pilot:** The orthogonal complement predicted emotional intensity level with 62% accuracy (chance=33%) and correlated with the empathy direction (cos ~0.3). The "extra dimensions" encode emotion-sensitive content.

In [ ]:
# === H11: Orthogonal Complement ===
from experiments.h11_orthogonal_complement import run as run_h11

h11_output = os.path.join(OUTPUT_DIR, "h11")
t0 = time.time()
run_h11(model, tokenizer, STIMULI_DIR, h11_output)
elapsed = time.time() - t0
print(f"\nH11 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h11_output, "h11_results.json"), meta)
show_plots(h11_output)

### H12: Activation Addition — Can We Override Sycophancy at Generation?

Add an anti-sycophancy vector at specific layers during generation. Measure the shift in log P(therapeutic) - log P(sycophantic).

**OLMo-2 1B pilot:** Steering at layer 10-12 with alpha=2.0 shifted log-prob ratio by +1.8 toward therapeutic. Effective intervention without retraining.

In [ ]:
# === H12: Activation Addition ===
from experiments.h12_activation_addition import run as run_h12

h12_output = os.path.join(OUTPUT_DIR, "h12")
t0 = time.time()
run_h12(model, tokenizer, STIMULI_DIR, h12_output, n_stimuli=N_STIMULI_H12)
elapsed = time.time() - t0
print(f"\nH12 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h12_output, "h12_results.json"), meta)
show_plots(h12_output)

### H13: Distortion x Emotion Interaction

H6 showed emotion modulates sycophancy. H7 showed distortion types vary. Do they interact?

**OLMo-2 1B pilot:** Catastrophizing (most factual-like) had a flat emotional gradient, while personalization (least factual-like) had a steep gradient. Emotion only reshapes the mechanism for distortions that aren't already factual-like.

In [ ]:
# === H13: Distortion x Emotion Interaction ===
from experiments.h13_distortion_emotion_interaction import run as run_h13

h13_output = os.path.join(OUTPUT_DIR, "h13")
t0 = time.time()
run_h13(model, tokenizer, STIMULI_DIR, h13_output)
elapsed = time.time() - t0
print(f"\nH13 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h13_output, "h13_results.json"), meta)
show_plots(h13_output)

### H14: Token Attribution — Which Prompt Tokens Trigger Sycophancy?

Gradient-based attribution: which input tokens most strongly activate the clinical sycophancy direction?

**OLMo-2 1B pilot:** Emotion words ("devastated", "terrified") were the strongest triggers, followed by first-person pronouns. Cognitive distortion markers ("always", "never") were less important than expected.

In [ ]:
# === H14: Token Attribution ===
from experiments.h14_token_attribution import run as run_h14

h14_output = os.path.join(OUTPUT_DIR, "h14")
t0 = time.time()
run_h14(model, tokenizer, STIMULI_DIR, h14_output, n_stimuli=N_STIMULI_H14)
elapsed = time.time() - t0
print(f"\nH14 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h14_output, "h14_results.json"), meta)
show_plots(h14_output)

### H15: Full Clinical Sycophancy Circuit Decomposition

Data-driven circuit analysis: logit lens on sycophancy and residual directions, layer-wise variance growth, per-stimulus residual clustering, and divergence mapping.

**OLMo-2 1B pilot:** Sycophancy direction pointed toward validation tokens ("understand", "valid") at late layers. Residual pointed toward hedging/dignity tokens ("however", "worth considering"). Divergence from factual sycophancy began at layer 8.

In [ ]:
# === H15: Circuit Decomposition ===
from experiments.h15_circuit_decomposition import run as run_h15

h15_output = os.path.join(OUTPUT_DIR, "h15")
t0 = time.time()
run_h15(model, tokenizer, STIMULI_DIR, h15_output)
elapsed = time.time() - t0
print(f"\nH15 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h15_output, "h15_results.json"), meta)
show_plots(h15_output)

### H16: Publication Validation Suite

Six validation experiments: behavioral sycophancy rate, generation with/without steering, steering generalization, random vector control, stimulus bootstrap, and residual direction validation.

**OLMo-2 1B pilot:** Behavioral sycophancy rate was 72% on clinical items. Steering reduced it to 41%. Random vectors had <5% effect, confirming specificity. Bootstrap CIs were tight on H1 and H10 findings.

In [ ]:
# === H16: Publication Validation ===
from experiments.h16_publication_validation import run as run_h16

h16_output = os.path.join(OUTPUT_DIR, "h16")
t0 = time.time()
run_h16(model, tokenizer, STIMULI_DIR, h16_output, n_generate=N_GENERATE_H16, max_new_tokens=30)
elapsed = time.time() - t0
print(f"\nH16 completed in {elapsed:.1f}s")
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
inject_metadata(os.path.join(h16_output, "h16_results.json"), meta)
show_plots(h16_output)

In [ ]:
# === Free DPO model before Phase 2 ===
print("Freeing DPO model...")
free_model(model)
del tokenizer
gc.collect()
if torch.cuda.is_available():
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")
print("DPO model freed. Ready for Phase 2.")

---
## Phase 2: Multi-Checkpoint Experiments

These experiments load base, SFT, and DPO checkpoints internally to track how training reshapes sycophancy circuits.

---

### H3: Does preference optimization conflate empathy and sycophancy?

Track empathy-sycophancy cosine similarity across base -> SFT -> DPO checkpoints.

**OLMo-2 1B pilot (base/instruct only):** Cosine similarity between empathy and sycophancy directions increased from 0.18 (base) to 0.35 (instruct), consistent with alignment training conflating the two.

In [ ]:
# === H3: Checkpoint Evolution ===
from experiments.h3_checkpoint_evolution import run as run_h3

h3_output = os.path.join(OUTPUT_DIR, "h3")
t0 = time.time()
run_h3(None, None, STIMULI_DIR, h3_output,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=QUANTIZE)
elapsed = time.time() - t0
print(f"\nH3 completed in {elapsed:.1f}s")
meta = build_metadata("multi-checkpoint", DEVICE)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
meta["checkpoints_used"] = CHECKPOINTS
inject_metadata(os.path.join(h3_output, "h3_results.json"), meta)
show_plots(h3_output)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### H5: DPO Reverses the Clinical Amplification Effect

Run the H2 logit lens analysis at each checkpoint to compare how the correct-answer signal changes across training stages.

**OLMo-2 1B pilot (base/instruct only):** Clinical correct-answer signal at peak layer dropped from +4.1 (base) to +3.1 (instruct). Alignment training suppresses the model's internal "I know the right answer" signal.

In [ ]:
# === H5: DPO Reversal ===
from experiments.h5_dpo_reversal import run as run_h5

h5_output = os.path.join(OUTPUT_DIR, "h5")
t0 = time.time()
run_h5(None, None, STIMULI_DIR, h5_output,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=QUANTIZE)
elapsed = time.time() - t0
print(f"\nH5 completed in {elapsed:.1f}s")
meta = build_metadata("multi-checkpoint", DEVICE)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
meta["checkpoints_used"] = CHECKPOINTS
inject_metadata(os.path.join(h5_output, "h5_results.json"), meta)
show_plots(h5_output)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### H8: Empathy-Sycophancy Entanglement Is Pre-Trained

Track empathy-sycophancy cosine similarity across checkpoints to test whether the entanglement originates during pretraining.

**OLMo-2 1B pilot (base/instruct only):** Empathy-sycophancy cosine similarity was already 0.18 in base model, suggesting pre-training origin. Alignment training amplified it to 0.35.

In [ ]:
# === H8: Pretraining Entanglement ===
from experiments.h8_pretraining_entanglement import run as run_h8

h8_output = os.path.join(OUTPUT_DIR, "h8")
t0 = time.time()
# For full pretraining trajectory, uncomment and add intermediate checkpoints:
# PRETRAINING_CHECKPOINTS = {
#     "step-10000": "allenai/Olmo-3-1025-7B:step-10000",
#     "step-50000": "allenai/Olmo-3-1025-7B:step-50000",
#     "step-100000": "allenai/Olmo-3-1025-7B:step-100000",
#     **CHECKPOINTS,
# }
run_h8(None, None, STIMULI_DIR, h8_output,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=QUANTIZE)
elapsed = time.time() - t0
print(f"\nH8 completed in {elapsed:.1f}s")
meta = build_metadata("multi-checkpoint", DEVICE)
meta["timestamp"] = datetime.now(timezone.utc).isoformat()
meta["runtime_s"] = round(elapsed, 1)
meta["checkpoints_used"] = CHECKPOINTS
inject_metadata(os.path.join(h8_output, "h8_results.json"), meta)
show_plots(h8_output)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Summary: Comprehensive Results

Load all 16 result JSONs and print a consolidated summary table with metadata.

In [ ]:
# === Consolidated Summary ===
import json
import os


hypotheses = [
    ("H1",  "Direction Comparison",           "h1/h1_results.json"),
    ("H2",  "Uncertainty vs Deference",       "h2/h2_results.json"),
    ("H3",  "Checkpoint Evolution",            "h3/h3_results.json"),
    ("H4",  "Attention Routing",               "h4/h4_results.json"),
    ("H5",  "DPO Reversal",                    "h5/h5_results.json"),
    ("H6",  "Emotional Gradient",              "h6/h6_results.json"),
    ("H7",  "Distortion Subspaces",            "h7/h7_results.json"),
    ("H8",  "Pretraining Entanglement",        "h8/h8_results.json"),
    ("H9",  "Warmth Tax",                      "h9/h9_results.json"),
    ("H10", "Extended Decomposition",          "h10/h10_results.json"),
    ("H11", "Orthogonal Complement",           "h11/h11_results.json"),
    ("H12", "Activation Addition",             "h12/h12_results.json"),
    ("H13", "Distortion x Emotion",            "h13/h13_results.json"),
    ("H14", "Token Attribution",               "h14/h14_results.json"),
    ("H15", "Circuit Decomposition",           "h15/h15_results.json"),
    ("H16", "Publication Validation",          "h16/h16_results.json"),
]

print("=" * 90)
print(f'{"Hyp":<5} {"Name":<30} {"Runtime":>10} {"Model":<30} {"Status"}')
print("=" * 90)

total_runtime = 0
for hid, name, rpath in hypotheses:
    full_path = os.path.join(OUTPUT_DIR, rpath)
    if os.path.exists(full_path):
        with open(full_path) as f:
            data = json.load(f)
        meta = data.get("metadata", {})
        runtime = meta.get("runtime_s", "?")
        model_name = meta.get("model", "?")
        if isinstance(runtime, (int, float)):
            total_runtime += runtime
            runtime_str = f"{runtime:.1f}s"
        else:
            runtime_str = str(runtime)
        # Truncate model name for display
        if len(model_name) > 28:
            model_name = "..." + model_name[-25:]
        print(f"{hid:<5} {name:<30} {runtime_str:>10} {model_name:<30} OK")
    else:
        print(f'{hid:<5} {name:<30} {"":>10} {"":<30} MISSING')

print("=" * 90)
print(f"Total runtime: {total_runtime:.1f}s ({total_runtime/60:.1f} min)")
print(f"Results directory: {OUTPUT_DIR}")

In [ ]:
# === Zip results for download ===
import shutil

model_tag = "olmo3_32b" if USE_32B else "olmo3_7b"
zip_path = f"/content/pals_{model_tag}_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"Results archived: {zip_path}.zip")
print(f"Size: {os.path.getsize(zip_path + ".zip") / 1e6:.1f} MB")

In [ ]:
# === Download results ===
try:
    from google.colab import files
    files.download("/content/pals_olmo3_7b_results.zip")
except ImportError:
    print("Not running in Colab. Download manually from: /content/pals_olmo3_7b_results.zip")